# AML GNN — Data Preparation v3: GFP Structural Features + FX-Corrected Amounts

Drop-in replacement for `Data_prepration.ipynb`.
Keeps the **original baseline edge/node features** from `Data_prepration.ipynb` (Section 1)
and appends **Graph Feature Preprocessor (GFP) structural features** from IBM SnapML (Section 2).

The only change to Section 1 is that `Amount_Log` now uses **day-specific FX rates** fetched
from Yahoo Finance so that amounts are truly comparable across the 7 currencies in LI-Small.

## Feature inventory (v3)

| Section | Group | Cols | Key AML signal |
|---|---|---|---|
| 1 | Transaction-level baseline (FX-corrected) | 16 | Amount in USD, OHE payment format, currency mismatch, timing, entity type |
| 2 | GFP: Fan in/out histogram | 4 | Fan-in/out degree buckets over time window |
| 2 | GFP: Degree histogram | 4 | In/out degree buckets |
| 2 | GFP: Scatter-Gather histogram | 2 | Scatter-gather pattern buckets |
| 2 | GFP: Temporal-cycle histogram | 2 | Cycle-closing edges in time window |
| 2 | GFP: Vertex stats (source, out) | ~8 | Fan, degree, ratio, avg, sum, var, skew, kurtosis of Amount USD |
| 2 | GFP: Vertex stats (source, in) | ~8 | Same stats for incoming edges |
| 2 | GFP: Vertex stats (dest, out) | ~8 | Same stats for destination outgoing |
| 2 | GFP: Vertex stats (dest, in) | ~8 | Same stats for destination incoming |

## Design principles

- **No data leakage:** GFP is fit on the full sorted transaction stream — it uses only
  transactions with timestamp < t for every feature of edge (u→v, t), by design of the library.
- **Original node features preserved:** `Bank_ID_Norm`, `EntityType` OHE (5 cols) — same as baseline.
- **Output files:** `edge_features_gfp.csv`, `node_features_gfp.csv`,
  `train/val/test_graph_gfp.pt`, `account_to_idx_gfp.pkl`.
- **EDGE_DIM** is printed at the end — update your model notebook accordingly.


---
## 0. Imports & Setup


In [5]:
# # Step 1: run this cell alone
# !pip install -q "numpy<2" snapml yfinance

In [6]:
# # Step 2: after restart, run this first
# import numpy
# print(numpy.__version__)  # must show 1.26.x before anything else loads

In [7]:
# from snapml import GraphFeaturePreprocessor
# print("GFP ready ✓")

In [8]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'numpy==1.26.4'], check=True)

# Now force reload numpy before any other import
import importlib, sys

# Remove any cached numpy from sys.modules
mods_to_remove = [k for k in sys.modules if 'numpy' in k]
for m in mods_to_remove:
    del sys.modules[m]

import numpy
print(numpy.__version__)  # should now show 1.26.4

1.26.4


/tmp/ipykernel_251/3438442988.py:12: UserWarning: The NumPy module was reloaded (imported a second time). This can in some cases result in small but subtle issues and is discouraged.
  import numpy


In [9]:
!pip install -q "numpy==1.26.4" snapml yfinance

In [10]:
# Try this first WITHOUT downgrading numpy
from snapml import GraphFeaturePreprocessor
gfp = GraphFeaturePreprocessor()
print("GFP ready ✓")

GFP ready ✓


In [11]:
!pip install torch_geometric -q

In [12]:
import psutil, os

# RAM
ram = psutil.virtual_memory()
print(f'RAM  : {ram.used/1e9:.1f} / {ram.total/1e9:.1f} GB  ({ram.percent}% used)')

# CPU
print(f'CPUs : {psutil.cpu_count()} cores')
print(f'CPU% : {psutil.cpu_percent(interval=1)}%')

# Disk
disk = psutil.disk_usage('/kaggle/working')
print(f'Disk : {disk.used/1e9:.1f} / {disk.total/1e9:.1f} GB used')

# Current session type
print(f'\nSession: {"GPU" if os.path.exists("/dev/nvidia0") else "CPU"}')

RAM  : 1.0 / 33.7 GB  (4.5% used)
CPUs : 4 cores
CPU% : 0.8%
Disk : 14.0 / 21.0 GB used

Session: CPU


In [13]:
import warnings, time, json, pickle
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import yfinance as yf

import torch
from torch_geometric.data import Data
from tqdm import tqdm

# SnapML Graph Feature Preprocessor
from snapml import GraphFeaturePreprocessor

class Timer:
    def __init__(self, label): self.label = label
    def __enter__(self): self.t = time.time(); return self
    def __exit__(self, *a): print(f'  [{self.label}] done in {time.time()-self.t:.1f}s')

print('Libraries loaded ✓')

Libraries loaded ✓


In [14]:
# # Mount Google Drive where the graph files and model checkpoints are stored
# from google.colab import drive
# import os

# drive.mount('/content/drive')

# os.chdir('/content/drive/MyDrive/GMA_GNN_AML')

In [15]:
# df_tr = pd.read_csv('Data/LI-Small_Trans.csv', low_memory=False)
# df_ac = pd.read_csv('Data/LI-Small_accounts.csv', low_memory=False)

df_tr = pd.read_csv('/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/LI-Small_Trans.csv', low_memory=False)
df_ac = pd.read_csv('/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/LI-Small_accounts.csv', low_memory=False)

print(f'Transactions : {len(df_tr):,}')
print(f'Accounts     : {len(df_ac):,}')
df_tr.head(3)

Transactions : 6,924,049
Accounts     : 712,688


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:08,11,8000ECA90,11,8000ECA90,3195403.00,US Dollar,3195403.00,US Dollar,Reinvestment,0
1,2022/09/01 00:21,3402,80021DAD0,3402,80021DAD0,1858.96,US Dollar,1858.96,US Dollar,Reinvestment,0
2,2022/09/01 00:00,11,8000ECA90,1120,8006AA910,592571.00,US Dollar,592571.00,US Dollar,Cheque,0


---
## 1. Transaction-level baseline features  *(v3: FX-corrected Amount_Log)*

Identical to the original `Data_prepration.ipynb` **except** that `Amount_Log` is computed
on the USD-converted amount rather than the raw `Amount Paid`.

**Why FX-correct the amount?**
LI-Small contains transactions in multiple currencies (USD, EUR, GBP, BTC, …).  
A raw `Amount Paid = 1000` means $1,000 if the currency is USD but ~$63M if it is Bitcoin.
Without conversion, the model sees structurally identical laundering patterns at wildly
different scales depending on the currency, making it impossible to learn a unified
amount-based signal.  Converting to USD first makes the distribution coherent.

**FX approach:** daily close rates fetched from Yahoo Finance for the exact date range
of the dataset (Sep 2022).  Weekends are forward-filled from the last trading close.
USD rows keep rate = 1.0 by definition.

All other features (OHE payment format, currency mismatch, time cyclicals, entity type, bank)
are unchanged from the baseline.


In [12]:
# ── 1.1  Fetch daily FX rates from Yahoo Finance ─────────────────────────────
# Map payment currency names to Yahoo Finance tickers.
# Crypto pairs use {ISO}-USD; FX pairs use {ISO}USD=X.
CURRENCY_TO_ISO = {
    'US Dollar':        'USD',
    'Euro':             'EUR',
    'British Pound':    'GBP',
    'Australian Dollar':'AUD',
    'Canadian Dollar':  'CAD',
    'Swiss Franc':      'CHF',
    'Chinese Yuan':     'CNY',
    'Japanese Yen':     'JPY',
    'Mexican Peso':     'MXN',
    'Brazilian Real':   'BRL',
    'Indian Rupee':     'INR',
    'South Korean Won': 'KRW',
    'Russian Ruble':    'RUB',
    'Saudi Riyal':      'SAR',
    'Singapore Dollar': 'SGD',
    'Hong Kong Dollar': 'HKD',
    'Norwegian Krone':  'NOK',
    'Swedish Krona':    'SEK',
    'Danish Krone':     'DKK',
    'New Zealand Dollar':'NZD',
    'South African Rand':'ZAR',
    'Turkish Lira':     'TRY',
    'UAE Dirham':       'AED',
    'Bitcoin':          'BTC',
    'Ethereum':         'ETH',
}

_all_currencies = df_tr['Payment Currency'].unique()
_non_usd        = [c for c in _all_currencies if c != 'US Dollar' and c in CURRENCY_TO_ISO]
print(f'Non-USD currencies in dataset: {_non_usd}')

# Date range: Sep 2022 with buffer.  Forward-fill covers weekends.
_START      = '2022-09-01'
_END        = '2022-09-19'
_full_dates = pd.date_range(start=_START, end='2022-09-17')

fx_rate_lookup = {}   # {(currency_name, 'YYYY-MM-DD'): rate_to_USD}

for _cname in _non_usd:
    _iso    = CURRENCY_TO_ISO[_cname]
    _ticker = f'{_iso}-USD' if _iso in ('BTC', 'ETH') else f'{_iso}USD=X'
    try:
        _raw = yf.download(_ticker, start=_START, end=_END,
                           auto_adjust=True, progress=False)
        if _raw.empty:
            print(f'  WARNING: no data for {_ticker}, defaulting to 1.0')
            continue
        _cl = _raw['Close']
        if hasattr(_cl, 'columns'):   # handle MultiIndex from some yfinance versions
            _cl = _cl.iloc[:, 0]
        _cl = _cl.squeeze()
        _cl = _cl.reindex(_full_dates).ffill().bfill()   # fill weekends
        for _dt, _rate in _cl.items():
            if pd.notna(_rate):
                fx_rate_lookup[(_cname, str(_dt.date()))] = float(_rate)
        print(f'  {_cname:25s} ({_ticker}): {_cl.notna().sum()} days, '
              f'last={float(_cl.iloc[-1]):.4f}')
    except Exception as _e:
        print(f'  ERROR fetching {_ticker}: {_e}')

print(f'\nFX lookup entries: {len(fx_rate_lookup):,}  (USD rows get rate=1.0 inline)')
print('FX rates loaded ✓')

Non-USD currencies in dataset: ['Euro', 'Bitcoin', 'Australian Dollar', 'Canadian Dollar', 'Mexican Peso', 'Swiss Franc', 'Saudi Riyal']
  Euro                      (EURUSD=X): 17 days, last=0.9988
  Bitcoin                   (BTC-USD): 17 days, last=20127.5762
  Australian Dollar         (AUDUSD=X): 17 days, last=0.6687
  Canadian Dollar           (CADUSD=X): 17 days, last=0.7549
  Mexican Peso              (MXNUSD=X): 17 days, last=0.0498
  Swiss Franc               (CHFUSD=X): 17 days, last=1.0397
  Saudi Riyal               (SARUSD=X): 17 days, last=0.2665

FX lookup entries: 119  (USD rows get rate=1.0 inline)
FX rates loaded ✓


In [13]:
# ── 1.2  Build edge dataframe with all baseline features ─────────────────────
edges = df_tr.copy()
edges['Timestamp'] = pd.to_datetime(edges['Timestamp'])
edges = edges.sort_values('Timestamp', kind='stable').reset_index(drop=True)

# Training cutoff (60%) — used for leakage-free target encodings later
_t1_idx = int(len(edges) * 0.60)

# ── Amount in USD + log1p ──────────────────────────────────────────────────
# Vectorised: build (currency, date) → rate via unique pairs, then left-merge
edges['_date_str'] = edges['Timestamp'].dt.date.astype(str)
_pairs = edges[['Payment Currency', '_date_str']].drop_duplicates().copy()
_pairs['_fx_rate'] = _pairs.apply(
    lambda r: fx_rate_lookup.get((r['Payment Currency'], r['_date_str']), 1.0),
    axis=1,
)
edges = edges.merge(_pairs, on=['Payment Currency', '_date_str'], how='left')
edges['_fx_rate']  = edges['_fx_rate'].fillna(1.0)
edges['Amount_USD'] = edges['Amount Paid'] * edges['_fx_rate']   # raw USD amount (used by GFP)
edges['Amount_Log'] = np.log1p(edges['Amount_USD'])               # log1p-compressed for GNN

# ── Currency mismatch ─────────────────────────────────────────────────────
edges['Currency_Mismatch'] = (
    edges['Receiving Currency'] != edges['Payment Currency']
).astype(np.int8)

# ── Cyclical time encoding ────────────────────────────────────────────────
_hour = edges['Timestamp'].dt.hour
_dow  = edges['Timestamp'].dt.dayofweek
edges['Hour_Sin']      = np.sin(2 * np.pi * _hour / 24)
edges['Hour_Cos']      = np.cos(2 * np.pi * _hour / 24)
edges['DayOfWeek_Sin'] = np.sin(2 * np.pi * _dow  / 7)
edges['DayOfWeek_Cos'] = np.cos(2 * np.pi * _dow  / 7)
edges['Is_Weekend']    = (_dow >= 5).astype(np.int8)

# ── Payment Format OHE  (7 dummies — same as baseline) ───────────────────
_fmt_dummies = pd.get_dummies(edges['Payment Format'], prefix='PayFmt').astype(np.int8)
edges = pd.concat([edges, _fmt_dummies], axis=1)
PAY_FMT_COLS = list(_fmt_dummies.columns)

# ── Is ACH flag ───────────────────────────────────────────────────────────
edges['Is_ACH'] = (edges['Payment Format'] == 'ACH').astype(np.int8)

# ── Self-loop ─────────────────────────────────────────────────────────────
edges['Is_Self_Loop'] = (edges['Account'] == edges['Account.1']).astype(np.int8)

# ── Rename for clarity ────────────────────────────────────────────────────
edges = edges.rename(columns={
    'Account':       'src_account',
    'Account.1':     'dst_account',
    'From Bank':     'src_bank',
    'To Bank':       'dst_bank',
    'Is Laundering': 'label',
})

# Baseline feature columns (same as original Data_prepration.ipynb
# except Amount_Log is now FX-corrected)
BASE_EDGE_COLS = (
    ['Amount_Log', 'Currency_Mismatch',
     'Hour_Sin', 'Hour_Cos', 'DayOfWeek_Sin', 'DayOfWeek_Cos',
     'Is_Weekend', 'Is_ACH', 'Is_Self_Loop']
    + PAY_FMT_COLS
)

print(f'Baseline edge features : {len(BASE_EDGE_COLS)}')
print(f'  {BASE_EDGE_COLS}')
print(f'\nAmount_USD stats:')
print(edges['Amount_USD'].describe().round(2))

Baseline edge features : 16
  ['Amount_Log', 'Currency_Mismatch', 'Hour_Sin', 'Hour_Cos', 'DayOfWeek_Sin', 'DayOfWeek_Cos', 'Is_Weekend', 'Is_ACH', 'Is_Self_Loop', 'PayFmt_ACH', 'PayFmt_Bitcoin', 'PayFmt_Cash', 'PayFmt_Cheque', 'PayFmt_Credit Card', 'PayFmt_Reinvestment', 'PayFmt_Wire']

Amount_USD stats:
count    6.924049e+06
mean     4.563286e+06
std      1.541978e+09
min      0.000000e+00
25%      2.136600e+02
50%      1.482220e+03
75%      1.199899e+04
max      3.644854e+12
Name: Amount_USD, dtype: float64


---
## 2. Node features  *(unchanged from baseline)*

- `Bank_ID_Norm`  — min-max normalised integer bank ID
- `EntityType` OHE — Corporation / Individual / Partnership / Sole Proprietor / Other (5 cols)

Keeping the original node features avoids the target-encoding instability identified
as the main cause of the v2 AUC regression (only 1,813 training positives → noisy rates).


In [14]:
nodes = df_ac.copy()
nodes = nodes.drop_duplicates(subset='Account Number', keep='first').reset_index(drop=True)

# ── Entity type ───────────────────────────────────────────────────────────
def _extract_entity_type(name):
    for t in ('Corporation', 'Individual', 'Partnership', 'Sole'):
        if t.lower() in str(name).lower():
            return t
    return 'Other'

nodes['Entity_Type'] = nodes['Entity Name'].apply(_extract_entity_type)

# ── Entity type OHE ───────────────────────────────────────────────────────
_et_dummies = pd.get_dummies(nodes['Entity_Type'], prefix='EntityType').astype(np.int8)
nodes = pd.concat([nodes, _et_dummies], axis=1)
ENTITY_TYPE_COLS = list(_et_dummies.columns)

# ── Bank ID normalised ────────────────────────────────────────────────────
_bmin, _bmax  = nodes['Bank ID'].min(), nodes['Bank ID'].max()
nodes['Bank_ID_Norm'] = (
    (nodes['Bank ID'] - _bmin) / (_bmax - _bmin + 1e-9)
).astype(np.float32)

NODE_FEAT_COLS = ['Bank_ID_Norm'] + ENTITY_TYPE_COLS
node_features  = nodes[['Account Number'] + NODE_FEAT_COLS].rename(
    columns={'Account Number': 'account_id'}
)

print(f'Node feature matrix: {node_features.shape}')
print(f'  Features: {NODE_FEAT_COLS}')
print(f'\nEntity type distribution:')
print(nodes['Entity_Type'].value_counts())

Node feature matrix: (712684, 6)
  Features: ['Bank_ID_Norm', 'EntityType_Corporation', 'EntityType_Individual', 'EntityType_Partnership', 'EntityType_Sole']

Entity type distribution:
Entity_Type
Corporation    268889
Partnership    225814
Sole           217130
Individual        851
Name: count, dtype: int64


---
## 3. Graph Feature Preprocessor (GFP) structural features

IBM SnapML's `GraphFeaturePreprocessor` computes the AML-specific graph patterns
described in Altman et al. Appendix D for every transaction in a single pass.

**Causal correctness:** GFP processes edges in timestamp order and uses only
edges with timestamp < t when computing features for edge (u→v, t).  
No leakage by construction.

### Patterns computed

| Pattern | Parameter | AML signal |
|---|---|---|
| Fan in/out | `fan`, bins=[2,3], window=24h | Fan-in mule accumulation, fan-out scatter |
| Degree in/out | `degree`, bins=[2,3], window=24h | Account activity intensity |
| Scatter-Gather | `scatter-gather`, bins=[2,3], window=6h | Rapid gather-then-scatter |
| Temporal cycle | `temp-cycle`, bins=[2,3], window=24h | Money loop closure |
| Vertex stats | source+dest, in+out, Amount_USD | Fan, degree, ratio, avg, sum, var, skew, kurtosis |

### Column naming convention

GFP returns a numpy array.  `build_gfp_colnames()` below reconstructs the
column names from the `params` dict — same logic as the reference notebook.


In [18]:
import json

# ── 3.1  GFP parameters (aligned with Altman et al. Appendix D) ──────────────
GFP_PARAMS = {
    'num_threads': 4,
    'time_window': 24 * 60 * 60,       # global fallback window (1 day)

    # ── Vertex statistics ─────────────────────────────────────────────────────
    # Paper Appendix D: "computed using the 'Amount' and 'Timestamp' fields"
    # GFP input layout:
    #   col 0 = transactionID
    #   col 1 = sourceAccountID
    #   col 2 = targetAccountID
    #   col 3 = timestamp (_ts_sec, relative seconds)
    #   col 4 = Amount_USD (FX-corrected)
    'vertex_stats':       True,
    'vertex_stats_tw':    24 * 60 * 60, # 1 day per paper (explicit)
    'vertex_stats_cols':  [3, 4],       # col3=timestamp, col4=Amount_USD
    'vertex_stats_feats': [0, 1, 2, 3, 4, 8, 9, 10],
    # 0:fan  1:degree  2:ratio  3:avg  4:sum  8:var  9:skew  10:kurtosis
    # NOTE: avg/sum of timestamps (col3) are noisy but
    #       var/skew/kurtosis capture burst timing patterns (strong AML signal)

    # ── Fan in/out — DISABLED ─────────────────────────────────────────────────
    # vertex_stats already gives raw continuous fan/degree values which are
    # strictly more informative than coarse histogram bins
    'fan':       False,
    'fan_bins':  [2, 5, 10],

    # ── Degree in/out — DISABLED ──────────────────────────────────────────────
    'degree':      False,
    'degree_bins': [2, 5, 10],

    # ── Scatter-Gather ────────────────────────────────────────────────────────
    # Paper: "time window of six hours for scatter-gather patterns"
    # bins: [2-3) minimal scatter, [3-5) moderate, [5+) highly suspicious
    'scatter-gather':      True,
    'scatter-gather_tw':   6 * 60 * 60,  # 6h per paper
    'scatter-gather_bins': [2, 3, 5],

    # ── Temporal cycle ────────────────────────────────────────────────────────
    # Paper: "time window of one day for the rest"
    # Fast detection of any cycle that closes within 24h
    # bins: [2-3) A→B→A round-trip, [3-5) 3-4 node ring, [5+) complex ring
    'temp-cycle':      True,
    'temp-cycle_tw':   24 * 60 * 60,     # 1 day per paper
    'temp-cycle_bins': [2, 3, 5],

    # ── Length-constrained simple cycle ──────────────────────────────────────
    # Paper: "simple cycles of length up to 10"
    # lc-cycle_len=6 is a speed/memory compromise vs paper's 10
    # bins must stay below lc-cycle_len: max bin boundary (5) < len (6) ✅
    # Set lc-cycle=False if Kaggle session runs out of time
    'lc-cycle':      True,
    'lc-cycle_tw':   24 * 60 * 60,       # 1 day per paper
    'lc-cycle_bins': [2, 3, 5],
    'lc-cycle_len':  6,                  # paper=10; 6 is safe compromise
}

print('GFP parameters:')
print(json.dumps(GFP_PARAMS, indent=4))


# ── Column name builder ───────────────────────────────────────────────────────
def build_gfp_colnames(params):
    """
    Reconstruct GFP output column names from params dict.

    GFP output column order per documentation:
      1. Pattern histograms (only enabled patterns):
            fan-in, fan-out, degree-in, degree-out,
            scatter-gather, temp-cycle, lc-cycle

      2. Vertex statistics (only if vertex_stats=True):
            For [source, dest] × [out, in]:
              fan, degree, ratio  (topology — once, not per column)
              for each col in vertex_stats_cols:
                avg, sum, [min, max, median,] var, skew, kurtosis
                (only feats listed in vertex_stats_feats)
    """
    colnames = []

    # ── 1. Pattern histogram columns ─────────────────────────────────────────
    for pattern in ['fan', 'degree', 'scatter-gather', 'temp-cycle', 'lc-cycle']:
        if not params.get(pattern, False):
            continue
        bins = params[pattern + '_bins']
        n    = len(bins)

        if pattern in ['fan', 'degree']:
            # fan and degree have separate in/out histograms
            for side in ['in', 'out']:
                for i in range(n - 1):
                    colnames.append(f'{pattern}_{side}_bins_{bins[i]}-{bins[i+1]}')
                colnames.append(f'{pattern}_{side}_bins_{bins[-1]}-inf')
        else:
            # scatter-gather, temp-cycle, lc-cycle: single histogram
            for i in range(n - 1):
                colnames.append(f'{pattern}_bins_{bins[i]}-{bins[i+1]}')
            colnames.append(f'{pattern}_bins_{bins[-1]}-inf')

    # ── 2. Vertex statistics columns ─────────────────────────────────────────
    if not params.get('vertex_stats', False):
        return colnames

    vert_feat_names = ['fan', 'deg', 'ratio',         # indices 0,1,2
                       'avg', 'sum', 'min', 'max',     # indices 3,4,5,6
                       'median', 'var', 'skew',        # indices 7,8,9
                       'kurtosis']                     # index 10

    col_labels = {3: 'ts', 4: 'amt'}   # human-readable suffix per col index

    # Order: source_out, source_in, dest_out, dest_in
    for orig in ['source', 'dest']:
        for direction in ['out', 'in']:

            # Topology stats: fan(0), degree(1), ratio(2) — once per direction
            for k in [0, 1, 2]:
                if k in params['vertex_stats_feats']:
                    colnames.append(f'{orig}_{vert_feat_names[k]}_{direction}')

            # Feature stats: one block per column in vertex_stats_cols
            for col in params['vertex_stats_cols']:
                label = col_labels.get(col, f'col{col}')
                for k in [3, 4, 5, 6, 7, 8, 9, 10]:
                    if k in params['vertex_stats_feats']:
                        colnames.append(
                            f'{orig}_{vert_feat_names[k]}_{label}_{direction}'
                        )

    return colnames


GFP_FEAT_COLS = build_gfp_colnames(GFP_PARAMS)

# ── Dimension verification ────────────────────────────────────────────────────
_enabled_patterns = ['scatter-gather', 'temp-cycle', 'lc-cycle']  # fan/degree disabled
_pat_total = sum(
    len(GFP_PARAMS[p + '_bins'])
    for p in _enabled_patterns
    if GFP_PARAMS.get(p, False)
)

_topo_per   = sum(1 for k in [0, 1, 2]
                  if k in GFP_PARAMS['vertex_stats_feats'])        # 3
_stat_per   = sum(1 for k in [3, 4, 5, 6, 7, 8, 9, 10]
                  if k in GFP_PARAMS['vertex_stats_feats'])        # 5
_n_cols     = len(GFP_PARAMS['vertex_stats_cols'])                 # 2
_vert_group = _topo_per + _stat_per * _n_cols                      # 3 + 5×2 = 13
_vert_total = 4 * _vert_group                                      # 4×13 = 52
_expected   = _pat_total + _vert_total

print(f'\n── GFP output dimensions ───────────────────────────────────')
for p in _enabled_patterns:
    if GFP_PARAMS.get(p, False):
        n = len(GFP_PARAMS[p + '_bins'])
        print(f'  {p:<22}: {n} cols  {GFP_PARAMS[p+"_bins"]}')
print(f'  {"─"*45}')
print(f'  pattern total          : {_pat_total} cols')
print(f'')
print(f'  topology per group     : {_topo_per}  (fan, deg, ratio)')
print(f'  stats per col          : {_stat_per}  (avg,sum,var,skew,kurt)')
print(f'  vertex_stats_cols      : {_n_cols}  ({GFP_PARAMS["vertex_stats_cols"]})')
print(f'  per group              : {_topo_per} + {_stat_per}×{_n_cols} = {_vert_group}')
print(f'  groups                 : 4  (src_out, src_in, dst_out, dst_in)')
print(f'  vertex stats total     : {_vert_total} cols')
print(f'  {"─"*45}')
print(f'  GFP TOTAL (expected)   : {_expected} cols')
print(f'  GFP TOTAL (function)   : {len(GFP_FEAT_COLS)} cols')

assert len(GFP_FEAT_COLS) == _expected, \
    f'MISMATCH: function={len(GFP_FEAT_COLS)}, expected={_expected}'
print(f'  Assertion ✓')

print(f'\nAll {len(GFP_FEAT_COLS)} GFP feature column names:')
for i, c in enumerate(GFP_FEAT_COLS):
    print(f'  [{i:02d}] {c}')

GFP parameters:
{
    "num_threads": 4,
    "time_window": 86400,
    "vertex_stats": true,
    "vertex_stats_tw": 86400,
    "vertex_stats_cols": [
        3,
        4
    ],
    "vertex_stats_feats": [
        0,
        1,
        2,
        3,
        4,
        8,
        9,
        10
    ],
    "fan": false,
    "fan_bins": [
        2,
        5,
        10
    ],
    "degree": false,
    "degree_bins": [
        2,
        5,
        10
    ],
    "scatter-gather": true,
    "scatter-gather_tw": 21600,
    "scatter-gather_bins": [
        2,
        3,
        5
    ],
    "temp-cycle": true,
    "temp-cycle_tw": 86400,
    "temp-cycle_bins": [
        2,
        3,
        5
    ],
    "lc-cycle": true,
    "lc-cycle_tw": 86400,
    "lc-cycle_bins": [
        2,
        3,
        5
    ],
    "lc-cycle_len": 6
}

── GFP output dimensions ───────────────────────────────────
  scatter-gather        : 3 cols  [2, 3, 5]
  temp-cycle            : 3 cols  [2, 3, 5]
  lc-cycle    

In [19]:
# ── 3.2  Build GFP input + memory cleanup ────────────────────────────────
import gc, psutil

# Build integer account index
_all_accs      = pd.concat([edges['src_account'], edges['dst_account']]).unique()
account_to_idx = {acc: idx for idx, acc in enumerate(_all_accs)}

_ts_epoch        = int(edges['Timestamp'].astype('int64').min() // 10**9)
edges['_ts_sec'] = (edges['Timestamp'].astype('int64') // 10**9) - _ts_epoch
edges['_src_idx'] = edges['src_account'].map(account_to_idx)
edges['_dst_idx'] = edges['dst_account'].map(account_to_idx)
edges['_txn_id']  = np.arange(len(edges))

# Convert to numpy immediately and drop the DataFrame copy — saves ~1-2 GB
gfp_input = edges[['_txn_id', '_src_idx', '_dst_idx', '_ts_sec', 'Amount_USD']]\
            .values.astype(np.float64)

# Slim edges down to only what Section 4 needs — free everything else
keep_cols  = ['src_account', 'dst_account', 'label', 'Timestamp'] + BASE_EDGE_COLS
edges      = edges[[c for c in keep_cols if c in edges.columns]].copy()

# Free raw dataframes and intermediate objects
del df_tr, df_ac, _all_accs
gc.collect()

ram = psutil.virtual_memory()
print(f'RAM after cleanup : {ram.used/1e9:.1f} / {ram.total/1e9:.1f} GB')
print(f'gfp_input shape   : {gfp_input.shape}  dtype={gfp_input.dtype}')
print(f'Accounts          : {len(account_to_idx):,}')

# ── 3.3  Run GFP ─────────────────────────────────────────────────────────
# GFP_PARAMS['num_threads'] = 2   # limit threads to reduce peak memory

print('\nInitialising GraphFeaturePreprocessor ...')
gfp = GraphFeaturePreprocessor()
gfp.set_params(GFP_PARAMS)

print(f'Running GFP on {gfp_input.shape[0]:,} edges ...')
print(f'  num_threads      = {GFP_PARAMS["num_threads"]}')
print(f'  Expected out cols: 5 + {len(GFP_FEAT_COLS)} = {5 + len(GFP_FEAT_COLS)}')
# print('  This may take 10–30 minutes ...')

with Timer('GFP fit_transform'):
    gfp_output = gfp.fit_transform(gfp_input)

# Free gfp_input immediately — no longer needed
del gfp_input, gfp
gc.collect()

print(f'\nGFP output shape: {gfp_output.shape}')
assert gfp_output.shape[0] == len(edges), 'Row count mismatch!'
assert gfp_output.shape[1] == 5 + len(GFP_FEAT_COLS), \
    f'Expected {5 + len(GFP_FEAT_COLS)} cols, got {gfp_output.shape[1]}'
print('Shape check ✓')

ram = psutil.virtual_memory()
print(f'RAM after GFP     : {ram.used/1e9:.1f} / {ram.total/1e9:.1f} GB')

RAM after cleanup : 3.4 / 33.7 GB
gfp_input shape   : (6924049, 5)  dtype=float64
Accounts          : 705,903

Initialising GraphFeaturePreprocessor ...
Running GFP on 6,924,049 edges ...
  num_threads      = 4
  Expected out cols: 5 + 61 = 66
  [GFP fit_transform] done in 2432.4s

GFP output shape: (6924049, 66)
Shape check ✓
RAM after GFP     : 9.8 / 33.7 GB


In [20]:
# ── 3.4  Extract GFP features into a DataFrame ───────────────────────────
# gfp_output columns: [txID, src, dst, ts, amount, gfp_feat_0, gfp_feat_1, ...]
# We skip the first 5 pass-through columns and keep only the new features.

gfp_feats_df = pd.DataFrame(
    gfp_output[:, 5:].astype(np.float32),
    columns=GFP_FEAT_COLS,
)

# Sanity: verify txID alignment (col 0 should be 0, 1, 2, ...)
_txn_ids = gfp_output[:, 0].astype(int)
assert (_txn_ids == np.arange(len(edges))).all(), \
    'Transaction IDs in GFP output are not aligned with edges DataFrame!'
print('Transaction ID alignment verified ✓')

print(f'\nGFP feature stats (first 5 columns):')
print(gfp_feats_df.iloc[:, :5].describe().round(3))

Transaction ID alignment verified ✓

GFP feature stats (first 5 columns):
       scatter-gather_bins_2-3  scatter-gather_bins_3-5  \
count              6924049.000                6924049.0   
mean                     0.000                      0.0   
std                      0.005                      0.0   
min                      0.000                      0.0   
25%                      0.000                      0.0   
50%                      0.000                      0.0   
75%                      0.000                      0.0   
max                      2.000                      0.0   

       scatter-gather_bins_5-inf  temp-cycle_bins_2-3  temp-cycle_bins_3-5  
count                  6924049.0           6924049.00          6924049.000  
mean                         0.0                 0.00                0.000  
std                          0.0                 0.02                0.003  
min                          0.0                 0.00                0.000  
25%      

---
## 4. Assemble final edge feature matrix & sanity checks


In [21]:
# ── 4.1  Concatenate baseline + GFP features ─────────────────────────────
EDGE_FEAT_COLS = BASE_EDGE_COLS + GFP_FEAT_COLS

edge_features = pd.concat([
    edges[['src_account', 'dst_account', 'label', 'Timestamp'] + BASE_EDGE_COLS]
        .reset_index(drop=True),
    gfp_feats_df.reset_index(drop=True),
], axis=1)

# Fill any NaN (first-transaction rows where GFP can't compute history)
nan_count = edge_features[EDGE_FEAT_COLS].isna().sum().sum()
edge_features[EDGE_FEAT_COLS] = edge_features[EDGE_FEAT_COLS].fillna(0).astype(np.float32)

print('── Feature inventory ──────────────────────────────────────────────')
print(f'  Baseline features  : {len(BASE_EDGE_COLS):>3}  (FX-corrected Amount_Log + OHE + timing)')
print(f'  GFP features       : {len(GFP_FEAT_COLS):>3}  (fan, degree, sg, temp-cycle, vertex stats)')
print(f'  ─────────────────────────────')
print(f'  TOTAL edge_attr dim: {len(EDGE_FEAT_COLS):>3}')
print(f'\nNaN values filled to 0: {nan_count:,}')
print(f'Edge feature matrix : {edge_features.shape}')

── Feature inventory ──────────────────────────────────────────────
  Baseline features  :  16  (FX-corrected Amount_Log + OHE + timing)
  GFP features       :  61  (fan, degree, sg, temp-cycle, vertex stats)
  ─────────────────────────────
  TOTAL edge_attr dim:  77

NaN values filled to 0: 0
Edge feature matrix : (6924049, 81)


In [22]:
# ── 4.2  Sanity check: laundering vs legitimate mean feature ratios ───────
signal_cols = [
    # Baseline
    'Amount_Log',
    'Currency_Mismatch',

    # Scatter-gather pattern (new bins)
    'scatter-gather_bins_2-3',
    'scatter-gather_bins_3-5',
    'scatter-gather_bins_5-inf',

    # Temporal cycle pattern (new bins)
    'temp-cycle_bins_2-3',
    'temp-cycle_bins_3-5',
    'temp-cycle_bins_5-inf',

    # LC cycle pattern
    'lc-cycle_bins_2-3',
    'lc-cycle_bins_3-5',
    'lc-cycle_bins_5-inf',

    # Vertex stats — topology (source)
    'source_fan_out',
    'source_deg_out',
    'source_ratio_out',
    'source_fan_in',
    'source_deg_in',

    # Vertex stats — topology (dest)
    'dest_fan_out',
    'dest_deg_out',
    'dest_fan_in',
    'dest_deg_in',

    # Vertex stats — amount (source outgoing)
    'source_avg_amt_out',
    'source_sum_amt_out',
    'source_var_amt_out',
    'source_skew_amt_out',

    # Vertex stats — amount (dest incoming)
    'dest_avg_amt_in',
    'dest_sum_amt_in',
    'dest_var_amt_in',

    # Vertex stats — timing variance (burst detection)
    'source_var_ts_out',
    'source_skew_ts_out',
    'dest_var_ts_in',
    'dest_skew_ts_in',
]

# Only check columns that actually exist
signal_cols = [c for c in signal_cols if c in edge_features.columns]
print(f'Checking {len(signal_cols)} signal columns...\n')

legit = edge_features[edge_features['label'] == 0][signal_cols]
laund = edge_features[edge_features['label'] == 1][signal_cols]
comp  = pd.DataFrame({
    'Legit mean'      : legit.mean(),
    'Laundering mean' : laund.mean(),
    'Ratio (L/l)'     : (laund.mean() / (legit.mean() + 1e-9)).round(2),
})
comp = comp.sort_values('Ratio (L/l)', ascending=False)
print('Sanity check — laundering vs legitimate feature means (ratio > 1 = good signal):')
print(comp.round(4).to_string())

# Quick summary
good_signals = (comp['Ratio (L/l)'] > 1.5).sum()
print(f'\nFeatures with ratio > 1.5 (strong signal): {good_signals} / {len(signal_cols)}')

Checking 31 signal columns...

Sanity check — laundering vs legitimate feature means (ratio > 1 = good signal):
                             Legit mean  Laundering mean  Ratio (L/l)
temp-cycle_bins_2-3        4.000000e-04     1.710000e-02    48.730000
lc-cycle_bins_2-3          4.000000e-04     1.710000e-02    48.669998
temp-cycle_bins_3-5        0.000000e+00     3.000000e-04    45.139999
scatter-gather_bins_2-3    0.000000e+00     6.000000e-04    28.760000
lc-cycle_bins_3-5          0.000000e+00     3.000000e-04    27.730000
dest_skew_ts_in           -3.170000e-02    -1.672000e-01     5.270000
source_fan_out             9.227574e+02     2.507132e+03     2.720000
source_deg_out             1.080253e+04     2.934324e+04     2.720000
source_fan_in              4.031050e+01     1.052354e+02     2.610000
source_sum_amt_out         1.002217e+10     2.501979e+10     2.500000
source_deg_in              9.205170e+01     2.140365e+02     2.330000
source_skew_amt_out        1.162050e+01     2.55

In [23]:
# ── 4.3  Save CSV files ───────────────────────────────────────────────────
import os
os.makedirs('/kaggle/working', exist_ok=True)

edge_features.to_csv('/kaggle/working/edge_features_gfp.csv', index=False)
node_features.to_csv('/kaggle/working/node_features_gfp.csv', index=False)

print('Saved:')
print(f'  /kaggle/working/edge_features_gfp.csv  — {edge_features.shape}')
print(f'  /kaggle/working/node_features_gfp.csv  — {node_features.shape}')
vc = edge_features['label'].value_counts()
print(f'\nClass balance: {vc[1]:,} laundering / {vc[0]:,} legitimate '
      f'({100*vc[1]/vc.sum():.4f}%)')

# Also save GFP_FEAT_COLS and EDGE_FEAT_COLS so you don't have to recompute
# them locally — just load this JSON and use it directly in the model notebook
import json
meta = {
    'GFP_FEAT_COLS':  GFP_FEAT_COLS,
    'BASE_EDGE_COLS': BASE_EDGE_COLS,
    'EDGE_FEAT_COLS': EDGE_FEAT_COLS,
    'NODE_FEAT_COLS': NODE_FEAT_COLS,
    'EDGE_DIM':       len(EDGE_FEAT_COLS),
    'NODE_DIM':       len(NODE_FEAT_COLS),
}
with open('/kaggle/working/feature_meta_gfp.json', 'w') as f:
    json.dump(meta, f, indent=2)

print(f'\nFeature metadata saved:')
print(f'  EDGE_DIM = {len(EDGE_FEAT_COLS)}')
print(f'  NODE_DIM = {len(NODE_FEAT_COLS)}')
print(f'  /kaggle/working/feature_meta_gfp.json')

Saved:
  /kaggle/working/edge_features_gfp.csv  — (6924049, 81)
  /kaggle/working/node_features_gfp.csv  — (712684, 6)

Class balance: 3,565 laundering / 6,920,484 legitimate (0.0515%)

Feature metadata saved:
  EDGE_DIM = 77
  NODE_DIM = 5
  /kaggle/working/feature_meta_gfp.json


---
## 5. Temporal 60 / 20 / 20 split


In [25]:
edge_df = edge_features.copy()
edge_df['Timestamp'] = pd.to_datetime(edge_df['Timestamp'])
edge_df = edge_df.sort_values('Timestamp', kind='stable').reset_index(drop=True)

n_edges = len(edge_df)
t1_idx  = int(n_edges * 0.60)
t2_idx  = int(n_edges * 0.80)

t1 = edge_df.loc[t1_idx - 1, 'Timestamp']
t2 = edge_df.loc[t2_idx - 1, 'Timestamp']

train_mask = edge_df.index < t1_idx
val_mask   = (edge_df.index >= t1_idx) & (edge_df.index < t2_idx)
test_mask  = edge_df.index >= t2_idx

print('Temporal split (60 / 20 / 20):')
print(f'  t1 (train end) : {t1}  →  {t1_idx:,} training edges')
print(f'  t2 (val end)   : {t2}  →  {t2_idx - t1_idx:,} validation edges')
print(f'  t_max          : {edge_df["Timestamp"].max()}  →  {n_edges - t2_idx:,} test edges')
print(f'\n  Laundering in train : {edge_df.loc[train_mask, "label"].sum():,} | Rate: {edge_df.loc[train_mask, "label"].mean()*100:.4f}%')
print(f'  Laundering in val   : {edge_df.loc[val_mask,   "label"].sum():,}   | Rate: {edge_df.loc[val_mask,   "label"].mean()*100:.4f}%')
print(f'  Laundering in test  : {edge_df.loc[test_mask,  "label"].sum():,}   | Rate: {edge_df.loc[test_mask,  "label"].mean()*100:.4f}%')

Temporal split (60 / 20 / 20):
  t1 (train end) : 2022-09-06 13:32:00  →  4,154,429 training edges
  t2 (val end)   : 2022-09-08 16:06:00  →  1,384,810 validation edges
  t_max          : 2022-09-17 15:28:00  →  1,384,810 test edges

  Laundering in train : 1,813 | Rate: 0.0436%
  Laundering in val   : 827   | Rate: 0.0597%
  Laundering in test  : 925   | Rate: 0.0668%


---
## 6. Graph construction

Same cumulative-snapshot protocol as `Data_prepration.ipynb`:
- **train_graph:** train edges only, all evaluated
- **val_graph:** train + val edges, evaluated on val portion
- **test_graph:** all edges, evaluated on test portion


In [ ]:
# ── 6.1  Node feature matrix ──────────────────────────────────────────────
all_accounts = pd.concat([
    node_features['account_id'],
    edge_df['src_account'],
    edge_df['dst_account'],
]).unique()

account_to_idx = {acc: idx for idx, acc in enumerate(all_accounts)}
N_nodes        = len(account_to_idx)
print(f'Total unique accounts (nodes): {N_nodes:,}')

nf_deduped = (
    node_features.drop_duplicates(subset='account_id', keep='first')
    if node_features['account_id'].duplicated().sum() > 0
    else node_features
)

idx_series    = pd.Series(account_to_idx)
node_feat_arr = (
    nf_deduped
    .set_index('account_id')
    .reindex(idx_series.index)
    [NODE_FEAT_COLS]
    .fillna(0)
    .values
    .astype(np.float32)
)
X_node = torch.tensor(node_feat_arr, dtype=torch.float)
print(f'Node feature matrix : {tuple(X_node.shape)}')

Total unique accounts (nodes): 712,684
Node feature matrix : (712684, 5)


In [ ]:
# ── 6.2  Graph builder function ───────────────────────────────────────────
def build_graph(edge_subset, eval_mask):
    """
    Build a PyG Data object from a subset of edges.

    Parameters
    ----------
    edge_subset : pd.DataFrame — rows from edge_df, reset_index applied
    eval_mask   : np.ndarray[bool] — which rows are in the evaluation set

    Returns
    -------
    torch_geometric.data.Data
    """
    src = edge_subset['src_account'].map(account_to_idx).values
    dst = edge_subset['dst_account'].map(account_to_idx).values

    edge_index = torch.tensor(np.stack([src, dst], axis=0), dtype=torch.long)
    edge_attr  = torch.tensor(
        edge_subset[EDGE_FEAT_COLS].values.astype(np.float32), dtype=torch.float
    )
    edge_time  = torch.tensor(
        edge_subset['Timestamp'].astype('int64').values // 10**9, dtype=torch.long
    )

    # Labels: -1 for context edges, 0/1 for evaluated edges
    # Use .values[eval_mask] — safe with boolean numpy array after reset_index
    labels             = np.full(len(edge_subset), -1, dtype=np.int64)
    labels[eval_mask]  = edge_subset['label'].values[eval_mask].astype(np.int64)

    return Data(
        x          = X_node,
        edge_index = edge_index,
        edge_attr  = edge_attr,
        edge_time  = edge_time,
        y          = torch.tensor(labels, dtype=torch.long),
        eval_mask  = torch.tensor(eval_mask, dtype=torch.bool),
        num_nodes  = N_nodes,
    )

In [ ]:
# ── 6.3  Build snapshots ──────────────────────────────────────────────────
print('Building graph snapshots ...')

with Timer('train graph'):
    train_edges = edge_df[train_mask].reset_index(drop=True)
    train_eval  = np.ones(len(train_edges), dtype=bool)
    train_graph = build_graph(train_edges, train_eval)

with Timer('val graph'):
    val_df      = edge_df[train_mask | val_mask].reset_index(drop=True)
    val_eval    = np.zeros(len(val_df), dtype=bool)
    val_eval[t1_idx:] = True
    val_graph   = build_graph(val_df, val_eval)

with Timer('test graph'):
    all_df      = edge_df.reset_index(drop=True)
    test_eval   = np.zeros(len(all_df), dtype=bool)
    test_eval[t2_idx:] = True
    test_graph  = build_graph(all_df, test_eval)

print('All snapshots built ✓')

Building graph snapshots ...
  [train graph] done in 7.1s
  [val graph] done in 14.1s
  [test graph] done in 21.0s
All snapshots built ✓


In [29]:
# # ── 6.4  Summary ─────────────────────────────────────────────────────────
# def summarise(name, g):
#     n_eval  = g.eval_mask.sum().item()
#     n_laund = (g.y[g.eval_mask] == 1).sum().item()
#     rate    = n_laund / n_eval * 100 if n_eval > 0 else 0
#     print(f'  {name:<14} | nodes={g.num_nodes:>7,} | edges={g.edge_index.shape[1]:>9,} '
#           f'| eval={n_eval:>9,} | laund={n_laund:>5,} ({rate:.4f}%)')

# print('\n── Graph snapshots ────────────────────────────────────────────────────')
# summarise('train_graph', train_graph)
# summarise('val_graph',   val_graph)
# summarise('test_graph',  test_graph)
# print(f'\nEdge feature dim : {train_graph.edge_attr.shape[1]}')
# print(f'Node feature dim : {train_graph.x.shape[1]}')

# ── 6.4  Summary ─────────────────────────────────────────────────────────
def summarise(name, g):
    n_eval  = g.eval_mask.sum().item()
    n_laund = (g.y[g.eval_mask] == 1).sum().item()
    rate    = n_laund / n_eval * 100 if n_eval > 0 else 0
    print(f'  {name:<14} | nodes={g.num_nodes:>7,} | '
          f'edges={g.edge_index.shape[1]:>9,} | '
          f'eval={n_eval:>9,} | laund={n_laund:>5,} ({rate:.4f}%)')

print('\n── Graph snapshots ──────────────────────────────────────────────')
summarise('train_graph', train_graph)
summarise('val_graph',   val_graph)
summarise('test_graph',  test_graph)
print(f'\nEdge feature dim : {train_graph.edge_attr.shape[1]}')
print(f'Node feature dim : {train_graph.x.shape[1]}')



── Graph snapshots ──────────────────────────────────────────────
  train_graph    | nodes=712,684 | edges=4,154,429 | eval=4,154,429 | laund=1,813 (0.0436%)
  val_graph      | nodes=712,684 | edges=5,539,239 | eval=1,384,810 | laund=  827 (0.0597%)
  test_graph     | nodes=712,684 | edges=6,924,049 | eval=1,384,810 | laund=  925 (0.0668%)

Edge feature dim : 77
Node feature dim : 5


In [30]:
# ── 6.5  Save ─────────────────────────────────────────────────────────────
torch.save(train_graph, '/kaggle/working/train_graph_gfp.pt')
torch.save(val_graph,   '/kaggle/working/val_graph_gfp.pt')
torch.save(test_graph,  '/kaggle/working/test_graph_gfp.pt')

with open('/kaggle/working/account_to_idx_gfp.pkl', 'wb') as f:
    pickle.dump(account_to_idx, f)

# Reload check
_g = torch.load('/kaggle/working/train_graph_gfp.pt', weights_only=False)
assert _g.edge_attr.shape[1] == len(EDGE_FEAT_COLS), \
    f'Feature dim mismatch: {_g.edge_attr.shape[1]} vs {len(EDGE_FEAT_COLS)}'

print('Saved:')
print('  /kaggle/working/train_graph_gfp.pt')
print('  /kaggle/working/val_graph_gfp.pt')
print('  /kaggle/working/test_graph_gfp.pt')
print('  /kaggle/working/account_to_idx_gfp.pkl')
print(f'\nReload check → edges: {_g.edge_index.shape[1]:,}  '
      f'edge_attr: {_g.edge_attr.shape}  ✓')
print()
print('═' * 60)
print(f'  ACTION: set  EDGE_DIM = {train_graph.edge_attr.shape[1]}')
print(f'          set  NODE_DIM = {train_graph.x.shape[1]}')
print(f'          in your model notebook')
print('═' * 60)

Saved:
  /kaggle/working/train_graph_gfp.pt
  /kaggle/working/val_graph_gfp.pt
  /kaggle/working/test_graph_gfp.pt
  /kaggle/working/account_to_idx_gfp.pkl

Reload check → edges: 4,154,429  edge_attr: torch.Size([4154429, 77])  ✓

════════════════════════════════════════════════════════════
  ACTION: set  EDGE_DIM = 77
          set  NODE_DIM = 5
          in your model notebook
════════════════════════════════════════════════════════════


---
## Appendix — Sanity checks


In [33]:
edge_features

,src_account,dst_account,label,Timestamp,Amount_Log,Currency_Mismatch,Hour_Sin,Hour_Cos,DayOfWeek_Sin,DayOfWeek_Cos,...,dest_avg_ts_in,dest_sum_ts_in,dest_var_ts_in,dest_skew_ts_in,dest_kurtosis_ts_in,dest_avg_amt_in,dest_sum_amt_in,dest_var_amt_in,dest_skew_amt_in,dest_kurtosis_amt_in
0,8000ECA90,8006AA910,0,2022-09-01 00:00:00,13.292228,0.0,0.000000,1.000000,0.433884,-0.900969,...,0.00000,0.0,0.000000e+00,0.000000,0.000000,592571.000000,5.925710e+05,0.000000e+00,0.000000,0.000000
1,8006AD530,8006AD530,0,2022-09-01 00:00:00,7.987035,0.0,0.000000,1.000000,0.433884,-0.900969,...,290290.71875,8128140.0,5.910173e+10,0.409444,2.086581,206.122147,5.771420e+03,3.003884e+05,4.442681,22.190268
2,800BC4DC0,80FE04360,0,2022-09-01 00:00:00,0.307485,0.0,0.000000,1.000000,0.433884,-0.900969,...,340766.40625,8519160.0,7.771156e+10,0.328874,1.717936,2215.954834,5.539887e+04,4.364924e+07,4.466665,21.583654
3,800F70C00,800F73990,0,2022-09-01 00:00:00,8.376318,0.0,0.000000,1.000000,0.433884,-0.900969,...,0.00000,0.0,0.000000e+00,0.000000,0.000000,4341.990234,4.341990e+03,0.000000e+00,0.000000,0.000000
4,800F77250,80CB23DC0,0,2022-09-01 00:00:00,4.953571,0.0,0.000000,1.000000,0.433884,-0.900969,...,356371.53125,27796980.0,7.450622e+10,0.260072,1.699017,7346.094238,5.729954e+05,2.585918e+09,8.541677,74.600388
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6924044,8001AB0D0,8001AB0D0,0,2022-09-16 13:24:00,9.436609,1.0,-0.258819,-0.965926,-0.433884,-0.900969,...,576294.37500,18441420.0,1.516205e+11,0.297810,2.176107,45269.902344,1.448637e+06,5.693836e+10,5.385005,30.010244
6924045,8001AB0D0,8011BC1F0,1,2022-09-16 13:24:00,9.596293,0.0,-0.258819,-0.965926,-0.433884,-0.900969,...,430114.06250,15914220.0,8.996863e+10,0.508411,3.373353,732.084595,2.708713e+04,7.591586e+06,4.288977,20.265202
6924046,8001F2990,8001F2990,0,2022-09-17 02:32:00,9.635366,1.0,0.500000,0.866025,-0.974928,-0.222521,...,596824.62500,15517440.0,1.835905e+11,0.363595,2.069490,24659.283203,6.411414e+05,1.192908e+10,4.784608,23.944483
6924047,8001F2990,80023D080,1,2022-09-17 02:32:00,9.475682,0.0,0.500000,0.866025,-0.974928,-0.222521,...,456861.59375,11421540.0,1.404504e+11,0.942564,3.584113,26696.177734,6.674044e+05,8.332452e+09,3.942030,17.617622


In [34]:
# Check 1: Amount_Log sanity (FX-corrected)
print('Check 1 — Amount_Log sanity (FX-corrected log1p of USD amount):')
print(edge_features['Amount_Log'].describe().round(3))
print(f'\nZero amounts     : {(edge_features["Amount_Log"] == 0).sum():,}')
print(f'Laundering mean  : {edge_features.loc[edge_features["label"]==1, "Amount_Log"].mean():.3f}')
print(f'Legitimate mean  : {edge_features.loc[edge_features["label"]==0, "Amount_Log"].mean():.3f}')

Check 1 — Amount_Log sanity (FX-corrected log1p of USD amount):
count    6924049.000
mean           7.546
std            3.137
min            0.000
25%            5.369
50%            7.302
75%            9.393
max           28.924
Name: Amount_Log, dtype: float64

Zero amounts     : 0
Laundering mean  : 8.518
Legitimate mean  : 7.545


In [36]:
# Check 2: GFP causal correctness
# First transaction per account must have zero history features
print('\nCheck 2 — GFP causal correctness:')
print('  First transaction per account should have 0 history stats ...')
first_txn_mask = edge_features.groupby('src_account').cumcount() == 0
history_cols   = [c for c in GFP_FEAT_COLS
                  if any(s in c for s in ['sum', 'avg', 'fan', 'deg'])]
history_cols   = history_cols[:5]   # sample of 5 cols
first_vals     = edge_features.loc[first_txn_mask, history_cols]
print(first_vals.describe().loc[['max']].round(4))
if (first_vals == 0).all().all():
    print('  All zero ✓  — GFP is causal, no future leakage')
else:
    print('  ⚠ Non-zero values detected — check GFP causal setting!')


Check 2 — GFP causal correctness:
  First transaction per account should have 0 history stats ...
     source_fan_out  source_deg_out  source_avg_ts_out  source_sum_ts_out  \
max         18942.0        222037.0         968776.375       9.146833e+10   

     source_avg_amt_out  
max        2.878440e+11  
  ⚠ Non-zero values detected — check GFP causal setting!


In [ ]:
# Check 3: Pearson correlation of ALL GFP features with label
print('\nCheck 3 — Pearson correlation of GFP features with label:')
corr_check = [c for c in GFP_FEAT_COLS if c in edge_features.columns] + ['label']
corr_series = (
    edge_features[corr_check]
    .corr()['label']
    .drop('label')
    .sort_values(ascending=False)
)
print('\nTop 10 positively correlated:')
print(corr_series.head(10).round(4).to_string())
print('\nTop 10 negatively correlated:')
print(corr_series.tail(10).round(4).to_string())
print(f'\nFeatures with |corr| > 0.01 : {(corr_series.abs() > 0.01).sum()} / {len(corr_series)}')


Check 3 — Pearson correlation of GFP features with label:


In [4]:
import shutil, os

# List files and total size
print('Files to zip:')
total = 0
for f in sorted(os.listdir('/kaggle/working')):
    size = os.path.getsize(f'/kaggle/working/{f}') / 1e6
    total += size
    print(f'  {f:<45} {size:.1f} MB')
print(f'\n  Total: {total/1000:.2f} GB')

# Zip directly — no copies needed
shutil.make_archive('/tmp/gfp_output', 'zip', '/kaggle/working')

# Move zip to working dir so it appears in Output panel
shutil.move('/tmp/gfp_output.zip', '/kaggle/working/gfp_output.zip')

zip_size = os.path.getsize('/kaggle/working/gfp_output.zip') / 1e9
print(f'\ngfp_output.zip created: {zip_size:.2f} GB ✓')

Files to zip:
  .virtual_documents                            0.0 MB
  account_to_idx_enhanced_gfp.pkl               11.9 MB
  account_to_idx_gfp.pkl                        12.0 MB
  edge_features_gfp.csv                         4298.7 MB
  feature_meta_gfp.json                         0.0 MB
  gfp_output.zip                                422.3 MB
  node_features_gfp.csv                         20.8 MB
  test_graph_gfp.pt                             2375.4 MB
  train_graph_gfp.pt                            1430.9 MB
  val_graph_gfp.pt                              1903.1 MB

  Total: 10.48 GB

gfp_output.zip created: 3.95 GB ✓


# Data Normalization

In [1]:
# ── Reload from saved CSV (after kernel restart) ──────────────────────────
import warnings, time, json, pickle
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data
from sklearn.preprocessing import RobustScaler, StandardScaler
from tqdm import tqdm

class Timer:
    def __init__(self, label): self.label = label
    def __enter__(self): self.t = time.time(); return self
    def __exit__(self, *a): print(f'  [{self.label}] done in {time.time()-self.t:.1f}s')

# ── Load feature metadata ─────────────────────────────────────────────────
with open('Data/feature_meta_gfp.json') as f:
    meta = json.load(f)

GFP_FEAT_COLS  = meta['GFP_FEAT_COLS']
BASE_EDGE_COLS = meta['BASE_EDGE_COLS']
EDGE_FEAT_COLS = meta['EDGE_FEAT_COLS']
NODE_FEAT_COLS = meta['NODE_FEAT_COLS']

print(f'EDGE_DIM : {len(EDGE_FEAT_COLS)}')
print(f'NODE_DIM : {len(NODE_FEAT_COLS)}')
print(f'GFP cols : {len(GFP_FEAT_COLS)}')
print(f'Base cols: {len(BASE_EDGE_COLS)}')

# ── Load edge and node features ───────────────────────────────────────────
print('\nLoading edge_features_gfp.csv ...')
with Timer('load edge_features'):
    edge_features = pd.read_csv('Data/edge_features_gfp.csv',
                                low_memory=False)
# edge_features['Timestamp'] = pd.to_datetime(edge_features['Timestamp'])
edge_features['Timestamp'] = pd.to_datetime(edge_features['Timestamp'], format='mixed')

print('\nLoading node_features_gfp.csv ...')
node_features = pd.read_csv('Data/node_features_gfp.csv',
                            low_memory=False)

print(f'\nedge_features : {edge_features.shape}')
print(f'node_features : {node_features.shape}')
vc = edge_features['label'].value_counts()
print(f'Class balance : {vc[1]:,} laundering / {vc[0]:,} legitimate '
      f'({100*vc[1]/vc.sum():.4f}%)')

EDGE_DIM : 77
NODE_DIM : 5
GFP cols : 61
Base cols: 16

Loading edge_features_gfp.csv ...
  [load edge_features] done in 225.2s

Loading node_features_gfp.csv ...

edge_features : (6924049, 81)
node_features : (712684, 6)
Class balance : 3,565 laundering / 6,920,484 legitimate (0.0515%)


In [2]:
edge_features.columns

Index(['src_account', 'dst_account', 'label', 'Timestamp', 'Amount_Log',
       'Currency_Mismatch', 'Hour_Sin', 'Hour_Cos', 'DayOfWeek_Sin',
       'DayOfWeek_Cos', 'Is_Weekend', 'Is_ACH', 'Is_Self_Loop', 'PayFmt_ACH',
       'PayFmt_Bitcoin', 'PayFmt_Cash', 'PayFmt_Cheque', 'PayFmt_Credit Card',
       'PayFmt_Reinvestment', 'PayFmt_Wire', 'scatter-gather_bins_2-3',
       'scatter-gather_bins_3-5', 'scatter-gather_bins_5-inf',
       'temp-cycle_bins_2-3', 'temp-cycle_bins_3-5', 'temp-cycle_bins_5-inf',
       'lc-cycle_bins_2-3', 'lc-cycle_bins_3-5', 'lc-cycle_bins_5-inf',
       'source_fan_out', 'source_deg_out', 'source_ratio_out',
       'source_avg_ts_out', 'source_sum_ts_out', 'source_var_ts_out',
       'source_skew_ts_out', 'source_kurtosis_ts_out', 'source_avg_amt_out',
       'source_sum_amt_out', 'source_var_amt_out', 'source_skew_amt_out',
       'source_kurtosis_amt_out', 'source_fan_in', 'source_deg_in',
       'source_ratio_in', 'source_avg_ts_in', 'source_sum_ts_in'

In [5]:
# select only kurtosis features in edge_features dataframe
kurt_cols = [c for c in edge_features.columns if 'kurt' in c]

edge_features[kurt_cols].describe()

,source_kurtosis_ts_out,source_kurtosis_amt_out,source_kurtosis_ts_in,source_kurtosis_amt_in,dest_kurtosis_ts_out,dest_kurtosis_amt_out,dest_kurtosis_ts_in,dest_kurtosis_amt_in
count,6.924049e+06,6.924049e+06,6.924049e+06,6.924049e+06,6.924049e+06,6.924049e+06,6.924049e+06,6.924049e+06
mean,1.590944e+00,1.561840e+03,1.317495e+00,6.477757e+01,1.093588e+00,1.794006e+01,1.832553e+00,1.323896e+01
std,3.976612e-01,6.815591e+03,8.743513e-01,2.549702e+02,7.799578e-01,5.737462e+02,2.824121e+00,2.515348e+01
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.556484e+00,2.188802e+00,1.031031e+00,1.339381e+00,0.000000e+00,0.000000e+00,1.513234e+00,2.135967e+00
50%,1.680311e+00,1.022271e+01,1.511532e+00,6.787407e+00,1.343174e+00,1.319608e+00,1.652196e+00,8.883286e+00
75%,1.749686e+00,3.402832e+01,1.694198e+00,1.979892e+01,1.662541e+00,7.420776e+00,1.752435e+00,2.069340e+01
max,1.107643e+01,3.725201e+04,8.491649e+01,1.433407e+03,1.107643e+01,3.725201e+04,8.491649e+01,1.433407e+03


In [2]:
# ── Temporal split indices ────────────────────────────────────────────────
edge_df = edge_features.sort_values('Timestamp', kind='stable').reset_index(drop=True)
n_edges = len(edge_df)
t1_idx  = int(n_edges * 0.60)
t2_idx  = int(n_edges * 0.80)

print(f'Split indices:')
print(f'  t1_idx (train end) : {t1_idx:,}')
print(f'  t2_idx (val end)   : {t2_idx:,}')
print(f'  test rows          : {n_edges - t2_idx:,}')



# ── Split GFP features ────────────────────────────────────────────────────
PATTERN_COLS = [c for c in GFP_FEAT_COLS if any(p in c for p in
                ['scatter-gather', 'temp-cycle', 'lc-cycle'])]
VERTEX_COLS  = [c for c in GFP_FEAT_COLS if c not in PATTERN_COLS]

# Further split vertex cols:
# LOG_COLS: non-negative, wide range → log1p first, then StandardScaler
# STD_COLS: can be negative (skew, kurtosis, ratio) → clip + StandardScaler only
LOG_COLS = [c for c in VERTEX_COLS if any(s in c for s in
            ['fan', 'deg', 'sum', 'avg', 'var', 'kurtosis'])]
STD_COLS = [c for c in VERTEX_COLS if c not in LOG_COLS]
# STD_COLS will contain: ratio, skew (can be negative)

print(f'Pattern cols  (no normalization)    : {len(PATTERN_COLS)}')
print(f'Log+Scale cols (log1p + StandardScaler): {len(LOG_COLS)}')
print(f'Scale-only cols (clip + StandardScaler): {len(STD_COLS)}')
print(f'  STD_COLS: {STD_COLS}')

# ── Helper: apply transformation to all splits ────────────────────────────
def transform_splits(train, val, test, p01, p99, scaler):
    tc = np.clip(train, p01, p99)
    vc = np.clip(val,   p01, p99)
    xc = np.clip(test,  p01, p99)
    scaler.fit(tc)
    return (scaler.transform(tc).astype(np.float32),
            scaler.transform(vc).astype(np.float32),
            scaler.transform(xc).astype(np.float32))

# ── Process LOG_COLS: log1p → clip → StandardScaler ──────────────────────
print(f'\nProcessing LOG_COLS ...')
train_log = edge_df.iloc[:t1_idx][LOG_COLS].values
val_log   = edge_df.iloc[t1_idx:t2_idx][LOG_COLS].values
test_log  = edge_df.iloc[t2_idx:][LOG_COLS].values

# log1p: safe for non-negative values, compresses heavy tail
# NOTE: var columns can be 0 → log1p(0) = 0, fine
train_log1p = np.log1p(np.clip(train_log, 0, None))  # clip negatives to 0 first
val_log1p   = np.log1p(np.clip(val_log,   0, None))
test_log1p  = np.log1p(np.clip(test_log,  0, None))

# Now clip at p01/p99 of the LOG-transformed training data
p01_log = np.percentile(train_log1p, 1,  axis=0)
p99_log = np.percentile(train_log1p, 99, axis=0)

scaler_log = StandardScaler()
tr_log_sc, va_log_sc, te_log_sc = transform_splits(
    train_log1p, val_log1p, test_log1p, p01_log, p99_log, scaler_log
)

# ── Process STD_COLS: clip → StandardScaler (no log, can be negative) ────
print(f'Processing STD_COLS ...')
train_std = edge_df.iloc[:t1_idx][STD_COLS].values
val_std   = edge_df.iloc[t1_idx:t2_idx][STD_COLS].values
test_std  = edge_df.iloc[t2_idx:][STD_COLS].values

p01_std = np.percentile(train_std, 1,  axis=0)
p99_std = np.percentile(train_std, 99, axis=0)

scaler_std = StandardScaler()
tr_std_sc, va_std_sc, te_std_sc = transform_splits(
    train_std, val_std, test_std, p01_std, p99_std, scaler_std
)

# ── Write back ────────────────────────────────────────────────────────────
edge_df.iloc[:t1_idx,
    edge_df.columns.get_indexer(LOG_COLS)] = tr_log_sc
edge_df.iloc[t1_idx:t2_idx,
    edge_df.columns.get_indexer(LOG_COLS)] = va_log_sc
edge_df.iloc[t2_idx:,
    edge_df.columns.get_indexer(LOG_COLS)] = te_log_sc

edge_df.iloc[:t1_idx,
    edge_df.columns.get_indexer(STD_COLS)] = tr_std_sc
edge_df.iloc[t1_idx:t2_idx,
    edge_df.columns.get_indexer(STD_COLS)] = va_std_sc
edge_df.iloc[t2_idx:,
    edge_df.columns.get_indexer(STD_COLS)] = te_std_sc

# Pattern bins untouched

# ── Verify ────────────────────────────────────────────────────────────────
print(f'\n── Post-normalization verification ──────────────────────────')
for name, s, e in [('Train', 0,      t1_idx),
                   ('Val',   t1_idx, t2_idx),
                   ('Test',  t2_idx, n_edges)]:
    v = edge_df.iloc[s:e][VERTEX_COLS].values
    print(f'  {name} vertex stats : min={v.min():>8.3f}  '
          f'max={v.max():>8.3f}  mean={v.mean():>7.3f}  std={v.std():>6.3f}')

print(f'\nPattern bins (unchanged):')
print(edge_df[PATTERN_COLS].describe().loc[['mean', 'max']].round(4))

# ── Save scalers ──────────────────────────────────────────────────────────
with open('Data/standard_scaler_gfp.pkl', 'wb') as f:
    pickle.dump({
        'scaler_log':   scaler_log,
        'scaler_std':   scaler_std,
        'log_cols':     LOG_COLS,
        'std_cols':     STD_COLS,
        'pattern_cols': PATTERN_COLS,
        'p01_log':      p01_log,
        'p99_log':      p99_log,
        'p01_std':      p01_std,
        'p99_std':      p99_std,
    }, f)

print('\nScalers saved → Data/standard_scaler_gfp.pkl ✓')
print('Normalization complete ✓')

Split indices:
  t1_idx (train end) : 4,154,429
  t2_idx (val end)   : 5,539,239
  test rows          : 1,384,810
Pattern cols  (no normalization)    : 9
Log+Scale cols (log1p + StandardScaler): 40
Scale-only cols (clip + StandardScaler): 12
  STD_COLS: ['source_ratio_out', 'source_skew_ts_out', 'source_skew_amt_out', 'source_ratio_in', 'source_skew_ts_in', 'source_skew_amt_in', 'dest_ratio_out', 'dest_skew_ts_out', 'dest_skew_amt_out', 'dest_ratio_in', 'dest_skew_ts_in', 'dest_skew_amt_in']

Processing LOG_COLS ...
Processing STD_COLS ...

── Post-normalization verification ──────────────────────────
  Train vertex stats : min=  -5.332  max=   5.051  mean=  0.000  std= 1.000
  Val vertex stats : min=  -4.689  max=   5.051  mean=  0.128  std= 0.896
  Test vertex stats : min=  -4.689  max=   5.051  mean=  0.061  std= 0.955

Pattern bins (unchanged):
      scatter-gather_bins_2-3  scatter-gather_bins_3-5  \
mean                      0.0                      0.0   
max                    

In [14]:
# ── Temporal split ────────────────────────────────────────────────────────
t1 = edge_df.loc[t1_idx - 1, 'Timestamp']
t2 = edge_df.loc[t2_idx - 1, 'Timestamp']

train_mask = edge_df.index < t1_idx
val_mask   = (edge_df.index >= t1_idx) & (edge_df.index < t2_idx)
test_mask  = edge_df.index >= t2_idx

print('Temporal split (60 / 20 / 20):')
print(f'  Train : {t1_idx:,} edges  | '
      f'laundering: {edge_df.loc[train_mask,"label"].sum():,}')
print(f'  Val   : {t2_idx-t1_idx:,} edges  | '
      f'laundering: {edge_df.loc[val_mask,"label"].sum():,}')
print(f'  Test  : {n_edges-t2_idx:,} edges  | '
      f'laundering: {edge_df.loc[test_mask,"label"].sum():,}')

# ── Node feature matrix ───────────────────────────────────────────────────
all_accounts = pd.concat([
    node_features['account_id'],
    edge_df['src_account'],
    edge_df['dst_account'],
]).unique()

account_to_idx = {acc: idx for idx, acc in enumerate(all_accounts)}
N_nodes        = len(account_to_idx)
print(f'\nTotal unique accounts (nodes): {N_nodes:,}')

nf_deduped = (
    node_features.drop_duplicates(subset='account_id', keep='first')
    if node_features['account_id'].duplicated().sum() > 0
    else node_features
)

idx_series    = pd.Series(account_to_idx)
node_feat_arr = (
    nf_deduped
    .set_index('account_id')
    .reindex(idx_series.index)
    [NODE_FEAT_COLS]
    .fillna(0)
    .values
    .astype(np.float32)
)
X_node = torch.tensor(node_feat_arr, dtype=torch.float)
print(f'Node feature matrix : {tuple(X_node.shape)}')

# ── Graph builder ─────────────────────────────────────────────────────────
def build_graph(edge_subset, eval_mask):
    src = edge_subset['src_account'].map(account_to_idx).values
    dst = edge_subset['dst_account'].map(account_to_idx).values

    edge_index = torch.tensor(np.stack([src, dst], axis=0), dtype=torch.long)
    edge_attr  = torch.tensor(
        edge_subset[EDGE_FEAT_COLS].values.astype(np.float32), dtype=torch.float
    )
    edge_time  = torch.tensor(
        edge_subset['Timestamp'].astype('int64').values // 10**9, dtype=torch.long
    )
    labels            = np.full(len(edge_subset), -1, dtype=np.int64)
    labels[eval_mask] = edge_subset['label'].values[eval_mask].astype(np.int64)

    return Data(
        x          = X_node,
        edge_index = edge_index,
        edge_attr  = edge_attr,
        edge_time  = edge_time,
        y          = torch.tensor(labels, dtype=torch.long),
        eval_mask  = torch.tensor(eval_mask, dtype=torch.bool),
        num_nodes  = N_nodes,
    )

# ── Build & save snapshots ────────────────────────────────────────────────
print('\nBuilding graph snapshots ...')

with Timer('train graph'):
    train_edges = edge_df[train_mask].reset_index(drop=True)
    train_graph = build_graph(train_edges, np.ones(len(train_edges), dtype=bool))

with Timer('val graph'):
    val_df      = edge_df[train_mask | val_mask].reset_index(drop=True)
    val_eval    = np.zeros(len(val_df), dtype=bool)
    val_eval[t1_idx:] = True
    val_graph   = build_graph(val_df, val_eval)


with Timer('test graph'):
    all_df      = edge_df.reset_index(drop=True)
    test_eval   = np.zeros(len(all_df), dtype=bool)
    test_eval[t2_idx:] = True
    test_graph  = build_graph(all_df, test_eval)

print('All snapshots built ✓')

# Summary
def summarise(name, g):
    n_eval  = g.eval_mask.sum().item()
    n_laund = (g.y[g.eval_mask] == 1).sum().item()
    rate    = n_laund / n_eval * 100 if n_eval > 0 else 0
    print(f'  {name:<14} | nodes={g.num_nodes:>7,} | '
          f'edges={g.edge_index.shape[1]:>9,} | '
          f'eval={n_eval:>9,} | laund={n_laund:>5,} ({rate:.4f}%)')

print('\n── Graph snapshots ──────────────────────────────────────────')
summarise('train_graph', train_graph)
summarise('val_graph',   val_graph)
summarise('test_graph',  test_graph)
print(f'\nEdge feature dim : {train_graph.edge_attr.shape[1]}')
print(f'Node feature dim : {train_graph.x.shape[1]}')

# Save
torch.save(train_graph, 'Data/train_graph_gfp.pt')
torch.save(val_graph,   'Data/val_graph_gfp.pt')
torch.save(test_graph,  'Data/test_graph_gfp.pt')

with open('Data/account_to_idx_gfp.pkl', 'wb') as f:
    pickle.dump(account_to_idx, f)

# Reload check
_g = torch.load('Data/train_graph_gfp.pt', weights_only=False)
assert _g.edge_attr.shape[1] == len(EDGE_FEAT_COLS)

print('\nSaved:')
print('  Data/train_graph_gfp.pt')
print('  Data/val_graph_gfp.pt')
print('  Data/test_graph_gfp.pt')
print('  Data/account_to_idx_gfp.pkl')
print(f'\nReload check ✓  edge_attr: {_g.edge_attr.shape}')
print()
print('═' * 60)
print(f'  EDGE_DIM = {train_graph.edge_attr.shape[1]}')
print(f'  NODE_DIM = {train_graph.x.shape[1]}')
print('═' * 60)

  [test graph] done in 27.8s
All snapshots built ✓

── Graph snapshots ──────────────────────────────────────────
  train_graph    | nodes=712,684 | edges=4,154,429 | eval=4,154,429 | laund=1,813 (0.0436%)
  val_graph      | nodes=712,684 | edges=5,539,239 | eval=1,384,810 | laund=  827 (0.0597%)
  test_graph     | nodes=712,684 | edges=6,924,049 | eval=1,384,810 | laund=  925 (0.0668%)

Edge feature dim : 77
Node feature dim : 5

Saved:
  Data/train_graph_gfp.pt
  Data/val_graph_gfp.pt
  Data/test_graph_gfp.pt
  Data/account_to_idx_gfp.pkl

Reload check ✓  edge_attr: torch.Size([4154429, 77])

════════════════════════════════════════════════════════════
  EDGE_DIM = 77
  NODE_DIM = 5
════════════════════════════════════════════════════════════
